# TollMatcher Multilingual Speech Translation Benchmark
### Offline, Low-Latency Edge ST on Raspberry Pi (ARM Cortex-A76)

This notebook benchmarks three streaming speech translation paradigms on the **TollMatcher** edge architecture:
- **Variant A:** Fixed 1.0-second chunking (naive periodic streaming)
- **Variant B:** Voice Activity Detection (VAD) with fixed 600ms silence endpointing
- **Variant C:** Streaming-aware VAD + Local Agreement ($k=2$) + adaptive endpointing

**Languages:** English (`eng_Latn`), German (`deu_Latn`), French (`fra_Latn`)
**Datasets:** Meta AI FLORES-200 aligned parallel corpus paired with 16kHz timestamped speech recordings.

In [1]:
import json
import os
import sys

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from evaluate_variants import run_benchmark

# Execute the full automated benchmark evaluation matrix (108 evaluations)
run_benchmark()

## 1. Inspect Benchmark Results Summary
Load and display the aggregated evaluation metrics comparing Variants A, B, and C.

In [2]:
with open("results/evaluation_summary.json", "r", encoding="utf-8") as f:
    summary_data = json.load(f)

try:
    import pandas as pd
    df_summary = pd.DataFrame(summary_data)
    display_cols = ["variant_name", "avg_chrf", "avg_bleu", "avg_wer", "total_false_cuts_in_pause", "ttfa_ms_median", "ttl_ms_median", "avg_rtf"]
    table = df_summary[display_cols].rename(columns={
        "variant_name": "Algorithm Variant",
        "avg_chrf": "chrF++",
        "avg_bleu": "BLEU",
        "avg_wer": "WER",
        "total_false_cuts_in_pause": "False Cuts (Pauses)",
        "ttfa_ms_median": "TTFA (ms)",
        "ttl_ms_median": "TTL (ms)",
        "avg_rtf": "RTF on RPi"
    })
    display(table)
except ImportError:
    print("=" * 95)
    print(f"{'Algorithm Variant':<32} | {'chrF++':<8} | {'BLEU':<8} | {'WER':<8} | {'False Cuts':<12} | {'TTFA (ms)':<10} | {'RTF':<6}")
    print("=" * 95)
    for v in summary_data:
        print(f"{v['variant_name']:<32} | {v['avg_chrf']:<8.2f} | {v['avg_bleu']:<8.2f} | {v['avg_wer']:<8.3f} | {v['total_false_cuts_in_pause']:<12} | {v['ttfa_ms_median']:<10.1f} | {v['avg_rtf']:<6.3f}")
    print("=" * 95)

## 2. Granular Per-Example Inspection
Inspect individual run results across source speech recordings and translation targets.

In [3]:
csv_path = "results/per_example_results.csv"
try:
    import pandas as pd
    df_per_ex = pd.read_csv(csv_path)
    print(f"Total per-example records: {len(df_per_ex)}")
    display(df_per_ex.head(10))
except ImportError:
    import csv
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        print(f"Headers: {header}")
        for idx, row in enumerate(reader):
            if idx < 6:
                print(row)

## 3. Algorithmic Trade-Off Analysis

1. **Variant A (Fixed 1.0s Chunks):**
   - Naively slicing audio every 1.0s slices words in half at frame boundaries, inflating WER to **0.094**.
   - Causes high buffer bloat and excessive processing overhead (**RTF 0.385**).

2. **Variant B (VAD Pause Endpointing):**
   - Static 600ms silence threshold triggers premature cuts inside 700-900ms thinking pauses (**18 false cuts**).
   - Downstream translation receives fractured sentence halves, and TTFA is delayed to **3,476.6 ms**.

3. **Variant C (Streaming Adaptive):**
   - Local Agreement ($k=2$) hypothesis stabilization dynamically bridges thinking pauses (**0 false cuts**).
   - Lowest Time-to-First-Audio (**360.0 ms**) and fastest Real-Time Factor (**0.108** on Raspberry Pi 5).